# Page Replacement Algorithm Simulator
**Operating Systems Lab Assignment | Virtual Memory / Page Replacement**  
**Algorithms:** FIFO, LRU and Optimal  
**Language:** Python 3 (Jupyter Notebook)

**Student:** ____________________  **Roll No.:** ____________________  
**Submission date:** 25 October 2026

## Objectives
- Simulate FIFO, LRU and Optimal page replacement.
- Show each page reference, frame contents, hit/fault status and replaced page.
- Calculate page faults, page hits, hit ratio and fault ratio.
- Compare all three algorithms and investigate Belady's anomaly.
- Complete the manual tracing and analysis sections from the assignment.

**Definitions:** A page hit means the referenced page is already in memory. A page fault means it is not in memory and must be loaded. Hit ratio = hits / total references; fault ratio = faults / total references.

In [1]:
# Page Replacement Algorithm Simulator
# Operating Systems Lab | Virtual Memory / Page Replacement
# Language: Python 3 (Jupyter Notebook)

from typing import List, Dict, Any, Tuple
import pandas as pd

def parse_reference_string(text: str) -> List[int]:
    """Parse comma- or whitespace-separated non-negative integer page numbers."""
    if not text or not text.strip():
        raise ValueError("Enter at least one page number.")
    tokens = text.replace(",", " ").split()
    pages = []
    for token in tokens:
        try:
            page = int(token)
        except ValueError:
            raise ValueError(
                "Page references must be whole numbers separated by commas or spaces."
            )
        if page < 0:
            raise ValueError("Page numbers must be non-negative integers.")
        pages.append(page)
    if not pages:
        raise ValueError("Enter at least one page number.")
    return pages

def simulate(reference: List[int], num_frames: int, algorithm: str) -> Tuple[int, int, List[Dict[str, Any]]]:
    """
    Simulate FIFO, LRU, or Optimal page replacement.
    Returns (page_faults, page_hits, trace).
    Trace frame lists use stable slot order. Empty slots are shown as '-'.
    """
    if not reference:
        raise ValueError("Reference string must contain at least one page.")
    if not isinstance(num_frames, int) or num_frames < 1:
        raise ValueError("Number of frames must be a whole number at least 1.")
    if any(not isinstance(p, int) or p < 0 for p in reference):
        raise ValueError("Page numbers must be non-negative integers.")

    policy = algorithm.strip().lower()
    if policy not in {"fifo", "lru", "optimal"}:
        raise ValueError("Algorithm must be FIFO, LRU, or Optimal.")

    frames = []
    arrival_order = []
    last_used = {}
    faults = 0
    hits = 0
    trace = []

    for i, page in enumerate(reference):
        replaced = "-"
        if page in frames:
            result = "H"
            hits += 1
            last_used[page] = i
        else:
            result = "F"
            faults += 1
            if len(frames) < num_frames:
                frames.append(page)
                arrival_order.append(page)
            else:
                if policy == "fifo":
                    victim = arrival_order.pop(0)
                elif policy == "lru":
                    victim = min(frames, key=lambda p: last_used[p])
                else:  # Optimal
                    future = reference[i + 1:]
                    distances = {}
                    for resident in frames:
                        try:
                            distances[resident] = future.index(resident)
                        except ValueError:
                            distances[resident] = float("inf")
                    # max() keeps the first frame on ties, including never-used-again pages.
                    victim = max(frames, key=lambda p: distances[p])
                replaced = victim
                slot = frames.index(victim)
                frames[slot] = page
                if policy == "fifo":
                    arrival_order.append(page)
                elif policy == "optimal":
                    pass
            last_used[page] = i

        # On FIFO, arrival_order is maintained independently of slot order.
        trace.append({
            "step": i + 1,
            "page": page,
            "frames": frames.copy(),
            "result": result,
            "replaced": replaced
        })

    return faults, hits, trace

def fifo(reference: List[int], num_frames: int):
    """FIFO wrapper; returns faults, hits and step-by-step trace."""
    return simulate(reference, num_frames, "fifo")

def lru(reference: List[int], num_frames: int):
    """LRU wrapper; returns faults, hits and step-by-step trace."""
    return simulate(reference, num_frames, "lru")

def optimal(reference: List[int], num_frames: int):
    """Optimal wrapper; returns faults, hits and step-by-step trace."""
    return simulate(reference, num_frames, "optimal")

def print_trace(reference: List[int], num_frames: int, algorithm: str) -> Dict[str, Any]:
    """Print a formatted trace and summary; return its metrics."""
    faults, hits, trace = simulate(reference, num_frames, algorithm)
    total = len(reference)
    print(f"\n{'=' * 18} {algorithm.upper()} {'=' * 18}")
    headers = ["Step", "Page", "Frame contents", "H/F", "Replaced"]
    rows = []
    for item in trace:
        frame_text = "[" + ", ".join(str(x) for x in item["frames"]) + "]"
        rows.append([item["step"], item["page"], frame_text, item["result"], item["replaced"]])
    display(pd.DataFrame(rows, columns=headers))
    print(f"Page faults : {faults}")
    print(f"Page hits   : {hits}")
    print(f"Hit ratio   : {hits / total:.2%}")
    print(f"Fault ratio : {faults / total:.2%}")
    return {
        "Algorithm": algorithm.upper(),
        "Page Faults": faults,
        "Page Hits": hits,
        "Hit Ratio": hits / total,
        "Fault Ratio": faults / total
    }

def compare_algorithms(reference: List[int], num_frames: int) -> pd.DataFrame:
    """Run all algorithms on identical input and display a comparison."""
    summaries = []
    for name in ("FIFO", "LRU", "Optimal"):
        summaries.append(print_trace(reference, num_frames, name))
    comparison = pd.DataFrame(summaries)
    comparison["Hit Ratio"] = comparison["Hit Ratio"].map(lambda x: f"{x:.2%}")
    comparison["Fault Ratio"] = comparison["Fault Ratio"].map(lambda x: f"{x:.2%}")
    print("\nFINAL PERFORMANCE COMPARISON")
    display(comparison)
    return comparison


## 1. Interactive input
Run this cell to enter your own page reference string. Pages may be separated by spaces or commas. Invalid input is caught and explained.

In [ ]:
try:
    raw_reference = input("Enter page reference string: ")
    reference = parse_reference_string(raw_reference)
    raw_frames = input("Enter number of frames: ")
    try:
        num_frames = int(raw_frames)
    except ValueError:
        raise ValueError("Number of frames must be a whole number.")
    if num_frames < 1:
        raise ValueError("Number of frames must be at least 1.")
    comparison = compare_algorithms(reference, num_frames)
except ValueError as error:
    print("Input error:", error)

## 2. Demonstration run
The sample below matches the assignment sample input: reference string `1 2 3 1 4 2 5`, with 3 frames. Run this cell to see all three traces and the final comparison.

In [3]:
sample_reference = [1, 2, 3, 1, 4, 2, 5]
sample_frames = 3
sample_comparison = compare_algorithms(sample_reference, sample_frames)


================== FIFO ==================


,Step,Page,Frame contents,H/F,Replaced
0,1,1,[1],F,-
1,2,2,"[1, 2]",F,-
2,3,3,"[1, 2, 3]",F,-
3,4,1,"[1, 2, 3]",H,-
4,5,4,"[4, 2, 3]",F,1
5,6,2,"[4, 2, 3]",H,-
6,7,5,"[4, 5, 3]",F,2


Page faults : 5
Page hits   : 2
Hit ratio   : 28.57%
Fault ratio : 71.43%

================== LRU ==================


,Step,Page,Frame contents,H/F,Replaced
0,1,1,[1],F,-
1,2,2,"[1, 2]",F,-
2,3,3,"[1, 2, 3]",F,-
3,4,1,"[1, 2, 3]",H,-
4,5,4,"[1, 4, 3]",F,2
5,6,2,"[1, 4, 2]",F,3
6,7,5,"[5, 4, 2]",F,1


Page faults : 6
Page hits   : 1
Hit ratio   : 14.29%
Fault ratio : 85.71%

================== OPTIMAL ==================


,Step,Page,Frame contents,H/F,Replaced
0,1,1,[1],F,-
1,2,2,"[1, 2]",F,-
2,3,3,"[1, 2, 3]",F,-
3,4,1,"[1, 2, 3]",H,-
4,5,4,"[4, 2, 3]",F,1
5,6,2,"[4, 2, 3]",H,-
6,7,5,"[5, 2, 3]",F,4


Page faults : 5
Page hits   : 2
Hit ratio   : 28.57%
Fault ratio : 71.43%

FINAL PERFORMANCE COMPARISON


,Algorithm,Page Faults,Page Hits,Hit Ratio,Fault Ratio
0,FIFO,5,2,28.57%,71.43%
1,LRU,6,1,14.29%,85.71%
2,OPTIMAL,5,2,28.57%,71.43%


## 3. Part B — Manual tracing verification
Reference string: `7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2`  
Frames: `3`

Run this cell and compare the trace with your hand-filled table.

In [4]:
part_b_reference = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]
part_b_comparison = compare_algorithms(part_b_reference, 3)


================== FIFO ==================


,Step,Page,Frame contents,H/F,Replaced
0,1,7,[7],F,-
1,2,0,"[7, 0]",F,-
2,3,1,"[7, 0, 1]",F,-
3,4,2,"[2, 0, 1]",F,7
4,5,0,"[2, 0, 1]",H,-
5,6,3,"[2, 3, 1]",F,0
6,7,0,"[2, 3, 0]",F,1
7,8,4,"[4, 3, 0]",F,2
8,9,2,"[4, 2, 0]",F,3
9,10,3,"[4, 2, 3]",F,0


Page faults : 10
Page hits   : 3
Hit ratio   : 23.08%
Fault ratio : 76.92%

================== LRU ==================


,Step,Page,Frame contents,H/F,Replaced
0,1,7,[7],F,-
1,2,0,"[7, 0]",F,-
2,3,1,"[7, 0, 1]",F,-
3,4,2,"[2, 0, 1]",F,7
4,5,0,"[2, 0, 1]",H,-
5,6,3,"[2, 0, 3]",F,1
6,7,0,"[2, 0, 3]",H,-
7,8,4,"[4, 0, 3]",F,2
8,9,2,"[4, 0, 2]",F,3
9,10,3,"[4, 3, 2]",F,0


Page faults : 9
Page hits   : 4
Hit ratio   : 30.77%
Fault ratio : 69.23%

================== OPTIMAL ==================


,Step,Page,Frame contents,H/F,Replaced
0,1,7,[7],F,-
1,2,0,"[7, 0]",F,-
2,3,1,"[7, 0, 1]",F,-
3,4,2,"[2, 0, 1]",F,7
4,5,0,"[2, 0, 1]",H,-
5,6,3,"[2, 0, 3]",F,1
6,7,0,"[2, 0, 3]",H,-
7,8,4,"[2, 4, 3]",F,0
8,9,2,"[2, 4, 3]",H,-
9,10,3,"[2, 4, 3]",H,-


Page faults : 7
Page hits   : 6
Hit ratio   : 46.15%
Fault ratio : 53.85%

FINAL PERFORMANCE COMPARISON


,Algorithm,Page Faults,Page Hits,Hit Ratio,Fault Ratio
0,FIFO,10,3,23.08%,76.92%
1,LRU,9,4,30.77%,69.23%
2,OPTIMAL,7,6,46.15%,53.85%


## 4. Part C — Experiment 1: Belady's anomaly
Reference string: `1 2 3 4 1 2 5 1 2 3 4 5`. Compare 3 frames with 4 frames.

In [5]:
belady_reference = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]
belady_rows = []
for frame_count in (3, 4):
    for name in ("FIFO", "LRU", "Optimal"):
        faults, hits, trace = simulate(belady_reference, frame_count, name)
        belady_rows.append({
            "Algorithm": name,
            "Frames": frame_count,
            "Page Faults": faults,
            "Page Hits": hits,
            "Hit Ratio": f"{hits / len(belady_reference):.2%}",
            "Fault Ratio": f"{faults / len(belady_reference):.2%}"
        })
belady_table = pd.DataFrame(belady_rows)
display(belady_table.pivot(index="Algorithm", columns="Frames", values="Page Faults").rename(columns={3: "Faults (3 frames)", 4: "Faults (4 frames)"}))
display(belady_table)

Frames,Faults (3 frames),Faults (4 frames)
Algorithm,,
FIFO,9,10
LRU,10,8
Optimal,7,6


,Algorithm,Frames,Page Faults,Page Hits,Hit Ratio,Fault Ratio
0,FIFO,3,9,3,25.00%,75.00%
1,LRU,3,10,2,16.67%,83.33%
2,Optimal,3,7,5,41.67%,58.33%
3,FIFO,4,10,2,16.67%,83.33%
4,LRU,4,8,4,33.33%,66.67%
5,Optimal,4,6,6,50.00%,50.00%


### Experiment 1 answers
1. **Does any algorithm produce more faults with more frames?** Yes. FIFO produces 9 faults with 3 frames and 10 faults with 4 frames. This is Belady's anomaly.
2. **Why do LRU and Optimal not show this behaviour?** LRU and Optimal are stack algorithms: the pages resident with *n* frames form a subset of those resident with *n + 1* frames. Adding a frame therefore cannot increase their page faults. FIFO does not have this stack property.

## 5. Part C — Experiment 2: Custom inputs
Two sample strings are included below. The first is designed so LRU has fewer faults than FIFO; the second is a tie, satisfying the assignment's “FIFO better than or equal to LRU” condition.

In [6]:
custom_cases = [
    ("LRU better", [1, 1, 1, 4, 3, 5, 5, 4, 3, 2, 2, 1, 4, 2, 4, 3, 2], 4),
    ("FIFO better/equal", [5, 3, 5, 2, 3, 1, 1, 2, 3, 5, 5, 2, 1, 3, 2, 3, 4, 1], 4),
]
custom_rows = []
for label, refs, frame_count in custom_cases:
    fifo_faults = fifo(refs, frame_count)[0]
    lru_faults = lru(refs, frame_count)[0]
    opt_faults = optimal(refs, frame_count)[0]
    custom_rows.append({
        "Case": label,
        "Reference string": " ".join(map(str, refs)),
        "Frames": frame_count,
        "FIFO faults": fifo_faults,
        "LRU faults": lru_faults,
        "Optimal faults": opt_faults
    })
custom_table = pd.DataFrame(custom_rows)
display(custom_table)

,Case,Reference string,Frames,FIFO faults,LRU faults,Optimal faults
0,LRU better,1 1 1 4 3 5 5 4 3 2 2 1 4 2 4 3 2,4,8,6,5
1,FIFO better/equal,5 3 5 2 3 1 1 2 3 5 5 2 1 3 2 3 4 1,4,5,5,5


## 6. Part C — Short questions
**1. Why can Optimal not be implemented in a real operating system?**  
It requires knowledge of future page references, which cannot be known in advance. It is used as a benchmark.

**2. What extra hardware or bookkeeping does LRU need compared with FIFO?**  
LRU must track when each page was last used, using timestamps/counters or a stack/list updated on accesses. FIFO only needs to maintain arrival order.

**3. If the number of frames equals the number of distinct pages in the string, how many faults will every algorithm produce?**  
Exactly one fault per distinct page. Once each distinct page is loaded, no page needs to be evicted, so later references are hits.

## 7. Conclusion
The simulator implements FIFO, LRU and Optimal with input validation, trace tables and performance metrics. The experiments show that Optimal provides the minimum number of faults for a given reference string and frame count, while FIFO can exhibit Belady's anomaly. LRU often benefits from locality of reference, but its performance depends on the access pattern.

## References
1. Silberschatz, Galvin and Gagne, *Operating System Concepts*, chapter on virtual memory and page replacement.
2. Operating Systems Lab Assignment — Page Replacement Algorithm Simulator.